# HW2 — Data Nature · Spatial-Statistical Model (Parts C–E)

Northern Israel NDVI / LST ecological monitoring.

**Pipeline:** GEE pixel export → PCA → Moran's I → spillover & PCA regression → Kriging → CA scenarios.

> Run cells top-to-bottom. The GEE export cell needs a one-time `ee.Authenticate()` (free Google Earth Engine account).

## 0 · Setup — install libraries & clone the repo

In [ ]:
!pip -q install earthengine-api scikit-learn statsmodels geopandas 2>/dev/null
![ -d data-nature ] || git clone -q https://github.com/AhmadTawil1/data-nature.git
import sys; sys.path.insert(0, 'data-nature/src')
print('ready')

## 1 · GEE pixel export  *(run once)*

Samples MODIS LST (`MOD11A1`) and NDVI (`MOD13Q1`) at 1 km over a 10 km buffer around each of the
8 sites, as the JuneAugust (summer) mean across 2024 and 2025. Saves a tidy pixel CSV. **Set your GEE project id below.**

In [ ]:
import ee, pandas as pd
ee.Authenticate()                 # one-time browser login
ee.Initialize(project='datanature')

# Summer mean: June+July+August, across BOTH 2024 and 2025
mfilt = ee.Filter.calendarRange(6, 8, 'month')
yfilt = ee.Filter.calendarRange(2024, 2025, 'year')
sites = pd.read_csv('data-nature/data/processed/site_locations.csv')

ndvi = (ee.ImageCollection('MODIS/061/MOD13Q1').filter(mfilt).filter(yfilt)
        .select('NDVI').mean().multiply(0.0001).rename('ndvi'))
lst  = (ee.ImageCollection('MODIS/061/MOD11A1').filter(mfilt).filter(yfilt)
        .select('LST_Day_1km').mean().multiply(0.02).subtract(273.15).rename('lst'))
img  = lst.addBands(ndvi).addBands(ee.Image.pixelLonLat())

rows = []
for _, r in sites.iterrows():
    region = ee.Geometry.Point([float(r.lng), float(r.lat)]).buffer(10000)
    feats = img.sample(region=region, scale=1000, geometries=False).getInfo()['features']
    for f in feats:
        p = f['properties']
        if p.get('lst') is None or p.get('ndvi') is None:
            continue
        rows.append(dict(lat=round(p['latitude'], 5), lng=round(p['longitude'], 5),
                         lst=round(p['lst'], 2), ndvi=round(p['ndvi'], 4),
                         site=r.site, land_cover=r.land_cover, month=7, year=2024))

pix = pd.DataFrame(rows)
pix.to_csv('pixels_summer_2024_2025.csv', index=False)
print(len(pix), 'pixels exported (JJA mean, 2024+2025) across', pix.site.nunique(), 'sites')
pix.head()

## 2 · Load data
If the GEE cell ran, this reads its output. Otherwise it falls back to the synthetic CA grid so the
notebook still runs end-to-end for review.

In [ ]:
import pandas as pd, numpy as np, os
from data_nature.stats import spatial as S

if os.path.exists('pixels_summer_2024_2025.csv'):
    df = pd.read_csv('pixels_summer_2024_2025.csv')
    print('Using real GEE pixels:', len(df))
else:
    print('GEE file not found — using synthetic CA-grid fallback.')
    from data_nature.models.optimization import init_site_grid
    md0 = pd.read_csv('data-nature/data/processed/site_monthly.csv')
    rng = np.random.default_rng(0); rows = []
    locs = pd.read_csv('data-nature/data/processed/site_locations.csv')
    for _, r in locs.iterrows():
        lstg, ndvg = init_site_grid(md0, r.site, 'Summer')
        for i in range(lstg.shape[0]):
            for j in range(lstg.shape[1]):
                rows.append(dict(lat=r.lat+(i-5)*0.003, lng=r.lng+(j-5)*0.003,
                                 lst=float(lstg[i,j]), ndvi=float(ndvg[i,j]),
                                 site=r.site, land_cover=r.land_cover, month=7, year=2024))
    df = pd.DataFrame(rows)

df = S.add_cyclic_month(df)
coords = df[['lat','lng']].values
df.head()

## 3 · PCA  (Part C / D)
Standardised PCA on the 6 numeric variables. We read off explained variance and the loadings to name
each component.

In [ ]:
scores, loadings, evr, pca_model = S.run_pca(df)
print('Explained variance ratio:'); print(evr.round(3))
print('\nLoadings (PC1, PC2):'); print(loadings[['PC1','PC2']].round(3))

### 3b · PCA biplot

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,6))
cats = df['land_cover'].astype('category')
sc = ax.scatter(scores[:,0], scores[:,1], c=cats.cat.codes, cmap='tab10', s=12, alpha=0.6)
for var in loadings.index:
    ax.arrow(0,0, loadings.loc[var,'PC1']*4, loadings.loc[var,'PC2']*4,
             color='k', head_width=0.08, length_includes_head=True)
    ax.text(loadings.loc[var,'PC1']*4.3, loadings.loc[var,'PC2']*4.3, var, fontsize=9)
ax.set_xlabel(f'PC1 ({evr.iloc[0]*100:.0f}%)'); ax.set_ylabel(f'PC2 ({evr.iloc[1]*100:.0f}%)')
ax.set_title('PCA biplot — pixels coloured by land cover'); ax.grid(alpha=0.3)
handles=[plt.Line2D([],[],marker='o',ls='',color=plt.cm.tab10(i/10),label=c)
         for i,c in enumerate(cats.cat.categories)]
ax.legend(handles=handles, fontsize=8, loc='best'); plt.tight_layout(); plt.show()

## 4 · Moran's I — spatial autocorrelation  (H1)
Tests whether LST is spatially clustered. Positive, significant I → reject H0₁.

In [ ]:
mi = S.morans_i(df['lst'].values, coords, k=8)
print(mi)
print('Reject H0_1 (LST spatially clustered)' if mi['p']<0.05 else 'Fail to reject H0_1')

## 5 · Regression — cooling effect & spatial spillover  (H2)
`pca_regression`: LST on principal components. `spillover_regression`: tests whether neighbour NDVI
cools a cell beyond its own NDVI.

In [ ]:
reg_pca = S.pca_regression(df, scores, k_pcs=2)
print('PCA regression:  R2=%.3f' % reg_pca.rsquared)
print(reg_pca.params.round(3).to_string(), '\n')

reg_sp = S.spillover_regression(df, coords, k=8)
print('Spillover regression:  R2=%.3f' % reg_sp.rsquared)
print('ndvi            coef=%.3f  p=%.4g' % (reg_sp.params['ndvi'], reg_sp.pvalues['ndvi']))
print('ndvi_neighbors  coef=%.3f  p=%.4g' % (reg_sp.params['ndvi_neighbors'], reg_sp.pvalues['ndvi_neighbors']))

## 6 · Kriging — spatial prediction  (H3)
Gaussian-process (Kriging) cross-validated against a global-mean baseline. Lower Kriging RMSE → LST
has exploitable spatial structure → reject H0₃.

In [ ]:
kr = S.kriging_cv(df, coords)
print(kr)
print('Reject H0_3 (spatial structure predictable)' if kr['kriging_rmse']<kr['mean_rmse'] else 'Fail to reject H0_3')

## 7 · Supplementary — ANOVA across land-cover types
Reuses the existing `regression.compare_site_types` from the repo.

In [ ]:
from data_nature.stats.regression import compare_site_types
res = compare_site_types(df)
print(res['anova_table'].to_string(index=False))
print('H0 rejected:', res['rejected_h0'])

## 8 · Results summary — fill these into the Part D text
Copy the printed numbers into the `[___]` placeholders in the HW2 document.

In [ ]:
print('Moran I  :', mi)
print('PCA reg  : R2=%.3f, PC1 coef=%.3f (p=%.3g)' % (reg_pca.rsquared, reg_pca.params['PC1'], reg_pca.pvalues['PC1']))
print('Spillover: ndvi=%.3f (p=%.3g), neighbors=%.3f (p=%.3g)' % (reg_sp.params['ndvi'], reg_sp.pvalues['ndvi'], reg_sp.params['ndvi_neighbors'], reg_sp.pvalues['ndvi_neighbors']))
print('Kriging  :', kr)
print('Variance :', dict(evr.round(3)))

---
# Part E — Simulation & Scenarios

Cellular-automaton heat-diffusion model (Game-of-Life-inspired). We run three scenarios on one site
and show how a vegetation change propagates spatially and over simulation steps.

## E1 · Cellular-automaton model

In [ ]:
import numpy as np, pandas as pd
GRID, COOLING, DIFF, RELAX = 10, 9.0, 0.30, 0.35
SEASON_HEAT = {'Spring':0.0,'Summer':0.4,'Autumn':-0.1,'Winter':-0.3}

def diffuse(lst):
    p = np.pad(lst, 1, mode='edge'); s = np.zeros_like(lst)
    for di in (-1,0,1):
        for dj in (-1,0,1):
            if di==0 and dj==0: continue
            s += p[1+di:GRID+1+di, 1+dj:GRID+1+dj]
    return (1-DIFF)*lst + DIFF*(s/8.0)

def run_sim(lst0, ndvi0, veg_pct, n=12, season='Summer'):
    ndvi = np.clip(ndvi0*(1+veg_pct/100.0), 0.01, 0.99)
    lst_eq = lst0 - COOLING*(ndvi-ndvi0); sh = SEASON_HEAT[season]*0.02
    lst = lst0.copy(); grids=[lst.copy()]; avgs=[float(lst.mean())]
    for _ in range(n):
        lst = lst + RELAX*(lst_eq-lst); lst = diffuse(lst); lst = lst + sh
        grids.append(lst.copy()); avgs.append(float(lst.mean()))
    return np.array(grids), np.array(avgs)

## E2 · Run the three scenarios
Baseline (no change) · Stress (−50% vegetation: drought/fire/urbanisation) · Recovery (+50%: replanting).

In [ ]:
from data_nature.models.optimization import init_site_grid
from data_nature.stats.spatial import morans_i

md_monthly = pd.read_csv('data-nature/data/processed/site_monthly.csv')
SITE = 'Carmel_Forest'
lst0, ndvi0 = init_site_grid(md_monthly, SITE, 'Summer')

ii, jj = np.meshgrid(range(GRID), range(GRID), indexing='ij')
grid_coords = np.c_[ii.ravel(), jj.ravel()]

scenarios = {'Baseline': 0, 'Stress (-50%)': -50, 'Recovery (+50%)': 50}
results = {}
for name, vp in scenarios.items():
    grids, avgs = run_sim(lst0, ndvi0, vp, n=12)
    mi = morans_i(grids[-1].ravel(), grid_coords, k=8, n_perm=199)['I']
    results[name] = dict(grids=grids, avgs=avgs, final=avgs[-1],
                         delta=avgs[-1]-avgs[0], moran=mi)
    print('%-16s final LST=%.2f  delta=%+.2f  Moran I=%.2f' % (name, avgs[-1], avgs[-1]-avgs[0], mi))

## E3 · Dashboard — scenario grids + LST trajectories

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 4, figsize=(17,4), gridspec_kw={'width_ratios':[1,1,1,1.3]})
vmin = min(r['grids'][-1].min() for r in results.values())
vmax = max(r['grids'][-1].max() for r in results.values())
for k,(name,r) in enumerate(results.items()):
    im = ax[k].imshow(r['grids'][-1], cmap='RdYlBu_r', vmin=vmin, vmax=vmax)
    ax[k].set_title(f"{name}\nmean {r['final']:.1f} C"); ax[k].set_xticks([]); ax[k].set_yticks([])
    fig.colorbar(im, ax=ax[k], fraction=0.046)
for name,r in results.items():
    ax[3].plot(r['avgs'], marker='o', ms=3, label=name)
ax[3].set_xlabel('simulation step'); ax[3].set_ylabel('avg LST (C)')
ax[3].set_title('LST trajectory'); ax[3].legend(fontsize=8); ax[3].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## E4 · Conclusions
Printed summary linking the scenarios back to the hypotheses, the PCA components, and the spatial patterns.

In [ ]:
b, s, r = results['Baseline'], results['Stress (-50%)'], results['Recovery (+50%)']
print('H2 (vegetation cooling): stress %+.2f C vs recovery %+.2f C -> vegetation drives LST' % (s['delta'], r['delta']))
print('H1 (spatial structure): Moran I stays ~%.2f -> heat remains spatially clustered in all scenarios' % np.mean([b['moran'],s['moran'],r['moran']]))
print('PCA: scenarios move along PC1 (thermal-vegetation) -> stress = hot/bare end, recovery = cool/green end')
print('Spatial pattern: vegetation change in the grid propagates to neighbours via diffusion (cooling/heating halo)')